# Notebook 13 — Cascaded Multi-Stage Ranking Pipeline

**Architecture:** Notebook 08 Hybrid Retrieval (100) → Notebook 09 LightGBM Pre-Ranking (N) → Notebook 10 DeepFM Fine-Ranking → Notebook 12 MMR (Top 10).

This notebook is a **faithful offline cascade replay** using the *previously exported per-candidate scores* for LightGBM and DeepFM. It gates the DeepFM stage to LightGBM Top-N, but **does not benchmark actual conditional model inference**; that is Notebook 15. The original DeepFM model was trained independently on Top-100 candidates, not specifically on pre-ranked Top-N.

**No leakage:** choose Top-N on the August user-disjoint holdout only. Evaluate September once after fixing N and MMR λ=0.9 from Notebook 12. Labels are used only for evaluation. The September models were already refit on full August by Notebooks 09 and 10.

**Data limitation:** RetailRocket has implicit views, not real ad impressions or auction outcomes. These are ranking scores, not calibrated CTR/CVR.

## 1. Setup and load identical cohorts

In [1]:
# pip install pandas numpy gensim scipy scikit-learn
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split

ROOT=Path.cwd().resolve()
if not (ROOT/'data'/'processed').is_dir(): ROOT=ROOT.parent
assert (ROOT/'data'/'processed').is_dir(), 'Run from project root or notebooks/'
PROC=ROOT/'data'/'processed'; REPORTS=ROOT/'reports'; MODELS=ROOT/'models'
REPORTS.mkdir(exist_ok=True)
SEED=42; TOP_N_GRID=(20,30,50,70,100); FINAL_K=10; MMR_LAMBDA=.9

def load_split(split):
    files={'hybrid':PROC/f'notebook08_{split}_hybrid_candidates.csv',
           'lgb':PROC/f'notebook09_{"val_holdout" if split=="val" else "test"}_scored_candidates.csv',
           'deepfm':PROC/f'notebook10_{"val_holdout" if split=="val" else "test"}_scored_candidates.csv'}
    for k,p in files.items(): assert p.exists(),f'Missing {k}: {p}'
    hybrid=pd.read_csv(files['hybrid'],usecols=['visitorid','itemid','rank'])
    lgb=pd.read_csv(files['lgb'],usecols=['visitorid','itemid','score','new_rank']).rename(columns={'score':'lgb_score','new_rank':'lgb_rank'})
    deepfm=pd.read_csv(files['deepfm'],usecols=['visitorid','itemid','score','new_rank']).rename(columns={'score':'deepfm_score','new_rank':'deepfm_rank'})
    assert not lgb[['visitorid','itemid']].duplicated().any()
    assert not deepfm[['visitorid','itemid']].duplicated().any()
    if split=='val':
        all_users=np.sort(hybrid.visitorid.unique())
        _,holdout_users=train_test_split(all_users,test_size=.20,random_state=SEED)
        cohort=set(map(int,holdout_users))
        hybrid=hybrid.loc[hybrid.visitorid.isin(cohort)]
    else: cohort=set(map(int,hybrid.visitorid.unique()))
    assert set(lgb.visitorid.unique())==set(deepfm.visitorid.unique())==cohort
    frame=hybrid.merge(lgb,on=['visitorid','itemid'],how='inner',validate='one_to_one')
    frame=frame.merge(deepfm,on=['visitorid','itemid'],how='inner',validate='one_to_one')
    assert len(frame)==len(hybrid)==len(lgb)==len(deepfm)
    assert frame.groupby('visitorid').size().eq(100).all()
    for col in ['rank','lgb_rank','deepfm_rank']:
        assert frame.groupby('visitorid')[col].nunique().eq(100).all(),col
    assert np.isfinite(frame[['lgb_score','deepfm_score']]).all().all()
    stem='validation' if split=='val' else 'test'
    labels=pd.read_csv(PROC/f'{stem}_view_labels_warm_users.csv',usecols=['visitorid','itemid']).drop_duplicates()
    labels=labels.loc[labels.visitorid.isin(cohort)]
    assert set(labels.visitorid.unique())==cohort
    vec_path=MODELS/f'notebook06_{split}_item2vec.model'
    assert vec_path.exists(),f'Missing Item2Vec model: {vec_path}'
    wv=Word2Vec.load(str(vec_path)).wv
    print(split,'users',len(cohort),'candidate rows',len(frame),'future positive pairs',len(labels))
    return frame,labels,wv

val,val_labels,val_wv=load_split('val')
test,test_labels,test_wv=load_split('test')

val users 2666 candidate rows 266600 future positive pairs 5206
test users 15137 candidate rows 1513700 future positive pairs 31876


## 2. Strict pre-ranking gate and DeepFM ordering

LightGBM keeps **Top N** only. DeepFM then sorts **only these N** candidates. The final 10 are selected with MMR, while other candidates never reach the fine-ranking stage. For N=100 the DeepFM ordering must match Notebook 10.

In [2]:
def cascade_pre_mmr(frame,top_n):
    assert 10<=top_n<=100
    gated=frame.loc[frame.lgb_rank.le(top_n)].copy()
    assert gated.groupby('visitorid').size().eq(top_n).all()
    gated=gated.sort_values(['visitorid','deepfm_score','lgb_rank','itemid'],ascending=[True,False,True,True],kind='mergesort')
    gated['fine_rank']=gated.groupby('visitorid',sort=False).cumcount()+1
    return gated

def evaluate_topk(frame,labels,rank_col,k=10):
    truth=labels.groupby('visitorid').itemid.agg(lambda s:set(map(int,s))).to_dict()
    ranked=frame.loc[frame[rank_col].le(k)]
    recs=ranked.groupby('visitorid').itemid.agg(list).to_dict()
    assert set(recs)==set(truth)
    denom=sum(map(len,truth.values())); hits=0; hit_users=0; macro=[]
    for u,items in recs.items():
        h=len(set(items)&truth[int(u)]);hits+=h;hit_users+=int(h>0);macro.append(h/len(truth[int(u)]))
    return {'Recall_micro':hits/denom,'Recall_macro':float(np.mean(macro)),
            'HitRate':hit_users/len(truth),'hits':hits}

# Sanity: ungated DeepFM Top-10 agrees with Notebook 10 exported ranking.
for f in (val,test):
    c=cascade_pre_mmr(f,100)
    original=f.loc[f.deepfm_rank.le(10),['visitorid','itemid']]
    replay=c.loc[c.fine_rank.le(10),['visitorid','itemid']]
    assert set(map(tuple,original.to_numpy()))==set(map(tuple,replay.to_numpy()))
print('PASS: N=100 reproduces DeepFM Top-10 and all Top-N gates have exact sizes')

PASS: N=100 reproduces DeepFM Top-10 and all Top-N gates have exact sizes


## 3. August holdout — choose the LightGBM gate

Choose the **smallest Top-N** within 99% of the *best observed August Recall@10* (tie-break with higher recall). This is a relevance-first proxy for lowering DeepFM scoring cost. MMR is held fixed at λ=0.9; its August tuning was already completed in Notebook 12. September is not consulted for this choice.

In [3]:
holdout_rows=[]
for top_n in TOP_N_GRID:
    g=cascade_pre_mmr(val,top_n)
    m=evaluate_topk(g,val_labels,'fine_rank',10)
    holdout_rows.append({'top_n':top_n,**m,'deepfm_scored_fraction':top_n/100})
holdout_table=pd.DataFrame(holdout_rows)
best=float(holdout_table.Recall_micro.max())
selected=holdout_table.loc[holdout_table.Recall_micro>=.99*best].sort_values(['top_n','Recall_micro'],ascending=[True,False]).iloc[0]
SELECTED_N=int(selected.top_n)
print('Selected LightGBM Top-N:',SELECTED_N,'holdout best Recall@10:',round(best,6))
display(holdout_table)

Selected LightGBM Top-N: 20 holdout best Recall@10: 0.035152


,top_n,Recall_micro,Recall_macro,HitRate,hits,deepfm_scored_fraction
0,20,0.035152,0.034870,0.058515,183,0.2
1,30,0.035152,0.034930,0.058515,183,0.3
2,50,0.034575,0.034694,0.058140,180,0.5
3,70,0.034191,0.034244,0.057389,178,0.7
4,100,0.034191,0.034244,0.057389,178,1.0


## 4. MMR on gated candidates (Top 10)

Cosine similarity from the **split-specific** Item2Vec model. Unknown item vectors receive zero similarity; diversity is reported on pairs where both embeddings are available. MMR uses min-max-normalized DeepFM scores *within each gated list*.

In [4]:
def mmr_order(items,scores,wv,lam=.9,k=10):
    items=np.asarray(items,dtype=np.int64);scores=np.asarray(scores,dtype=np.float64)
    n=len(items);assert n>=k
    lo,hi=float(scores.min()),float(scores.max())
    rel=(scores-lo)/(hi-lo) if hi>lo else np.zeros(n,dtype=np.float64)
    vectors=np.zeros((n,wv.vector_size),dtype=np.float32);known=np.zeros(n,dtype=bool)
    for j,item in enumerate(items):
        key=str(int(item))
        if key in wv: vectors[j]=wv[key];known[j]=True
    norm=np.linalg.norm(vectors,axis=1,keepdims=True)
    np.divide(vectors,np.maximum(norm,1e-12),out=vectors)
    sim=np.clip(vectors@vectors.T,0,1);np.fill_diagonal(sim,0)
    remaining=np.ones(n,dtype=bool);redundancy=np.zeros(n);chosen=[]
    for _ in range(k):
        scores_mmr=lam*rel-(1-lam)*redundancy
        scores_mmr[~remaining]=-np.inf
        j=int(np.argmax(scores_mmr));chosen.append(j);remaining[j]=False
        redundancy=np.maximum(redundancy,sim[:,j])
    known_selected=np.array(chosen,dtype=int)[known[chosen]]
    if len(known_selected)>1:
        sub=sim[np.ix_(known_selected,known_selected)]
        tri=sub[np.triu_indices(len(known_selected),1)]
        pair_sum=float(tri.sum());pairs=len(tri)
    else:pair_sum=0.;pairs=0
    return chosen,{'known_pair_sum':pair_sum,'known_pairs':pairs,'embedded_items':int(known[chosen].sum())}

def rerank_mmr(gated,wv,lam=.9):
    outputs=[];stats=[]
    for u,part in gated.groupby('visitorid',sort=False):
        ordered=part.sort_values(['fine_rank'],kind='mergesort')
        ix,stat=mmr_order(ordered.itemid.to_numpy(),ordered.deepfm_score.to_numpy(),wv,lam,FINAL_K)
        out=ordered.iloc[ix].copy();out['final_rank']=np.arange(1,FINAL_K+1,dtype=np.int16)
        outputs.append(out);stats.append(stat)
    return pd.concat(outputs,ignore_index=True),pd.DataFrame(stats)

def diversity(stats):
    total_pairs=int(stats.known_pairs.sum())
    return {'known_pair_ILD':1-float(stats.known_pair_sum.sum()/total_pairs) if total_pairs else None,
            'known_pair_fraction':total_pairs/(45*len(stats)),
            'mean_embedded_items_at_10':float(stats.embedded_items.mean())}

# Small deterministic unit test: identical vectors should incur a redundancy penalty.
class Toy:
    vector_size=2
    def __contains__(self,k):return k in {'1','2','3'}
    def __getitem__(self,k):return {'1':np.array([1.,0.]),'2':np.array([1.,0.]),'3':np.array([0.,1.])}[k]
assert mmr_order([1,2,3],[.9,.8,.7],Toy(),1.0,3)[0]==[0,1,2]
print('PASS: MMR ordering unit test')

PASS: MMR ordering unit test


## 5. Evaluate chosen cascade on August, then September **once**

All tuning choices are now frozen: LightGBM Top-N from August; MMR λ=0.9 from Notebook 12. Report the ungated DeepFM baseline, gated DeepFM, and gated DeepFM+MMR.

In [5]:
def evaluate_split(frame,labels,wv,top_n):
    t0=time.perf_counter()
    ungated=cascade_pre_mmr(frame,100)
    gated=cascade_pre_mmr(frame,top_n)
    final,stats=rerank_mmr(gated,wv,MMR_LAMBDA)
    metrics={'ungated_deepfm':evaluate_topk(ungated,labels,'fine_rank'),
             'gated_deepfm':evaluate_topk(gated,labels,'fine_rank'),
             'gated_deepfm_mmr':evaluate_topk(final,labels,'final_rank')}
    return gated,final,metrics,diversity(stats),time.perf_counter()-t0

val_gated,val_final,val_metrics,val_div,val_elapsed=evaluate_split(val,val_labels,val_wv,SELECTED_N)
print('August holdout metrics:',json.dumps(val_metrics,indent=2))
# September test is evaluated once after the gate and lambda are frozen.
test_gated,test_final,test_metrics,test_div,test_elapsed=evaluate_split(test,test_labels,test_wv,SELECTED_N)
print('September test metrics:',json.dumps(test_metrics,indent=2))
print('September diversity:',test_div)
assert val_final.groupby('visitorid').size().eq(10).all()
assert test_final.groupby('visitorid').size().eq(10).all()
assert not test_final[['visitorid','itemid']].duplicated().any()
assert test_final.final_rank.max()==10
# Confirm the ungated baseline matches Notebook 10's previously reported result.
assert np.isclose(test_metrics['ungated_deepfm']['Recall_micro'],0.02710503199899611,atol=1e-9)
print('PASS: cohort, output uniqueness, baseline reproduction, and final Top-10 invariants')

August holdout metrics: {
  "ungated_deepfm": {
    "Recall_micro": 0.03419131771033423,
    "Recall_macro": 0.03424382410311877,
    "HitRate": 0.05738934733683421,
    "hits": 178
  },
  "gated_deepfm": {
    "Recall_micro": 0.03515174798309643,
    "Recall_macro": 0.034870206188836274,
    "HitRate": 0.058514628657164294,
    "hits": 183
  },
  "gated_deepfm_mmr": {
    "Recall_micro": 0.03553592009220131,
    "Recall_macro": 0.03537033122009409,
    "HitRate": 0.05926481620405101,
    "hits": 185
  }
}
September test metrics: {
  "ungated_deepfm": {
    "Recall_micro": 0.02710503199899611,
    "Recall_macro": 0.030666360117872795,
    "HitRate": 0.04849045385479289,
    "hits": 864
  },
  "gated_deepfm": {
    "Recall_micro": 0.027356004517505333,
    "Recall_macro": 0.030951924955615467,
    "HitRate": 0.04901896016383696,
    "hits": 872
  },
  "gated_deepfm_mmr": {
    "Recall_micro": 0.027387376082318986,
    "Recall_macro": 0.03090159102142079,
    "HitRate": 0.049018960163836

## 6. Export pipeline artifacts and JSON report

In [6]:
val_final.to_csv(PROC/'notebook13_val_holdout_cascade_top10.csv',index=False)
test_final.to_csv(PROC/'notebook13_test_cascade_top10.csv',index=False)
report={'method':'offline cascaded ranking replay with precomputed stage scores',
        'pipeline':['Notebook08 Hybrid Retrieval Top100','Notebook09 LightGBM gate TopN',
                    'Notebook10 DeepFM order gated candidates','Notebook12 MMR final Top10'],
        'protocol':'August holdout selects TopN; MMR lambda fixed from Notebook12 August; September test once',
        'candidate_pool':100,'top_n_grid':list(TOP_N_GRID),'selected_top_n':SELECTED_N,
        'mmr_lambda':MMR_LAMBDA,'holdout_gate_sweep':holdout_table.to_dict(orient='records'),
        'holdout':val_metrics,'holdout_diversity':val_div,
        'test':test_metrics,'test_diversity':test_div,
        'test_replay_seconds':test_elapsed,
        'estimated_deepfm_candidate_reduction_fraction':1-SELECTED_N/100,
        'caveats':['Precomputed full-100 LightGBM/DeepFM scores: gating is simulated, not actual conditional inference',
                   'Test replay runtime is not p50/p95 serving latency',
                   'DeepFM was trained independently on Top100, not specifically on the LightGBM TopN distribution',
                   'No impression/auction logs; implicit negatives are not confirmed negatives',
                   'Item2Vec missing for some items; known-pair ILD is coverage-sensitive']}
path=REPORTS/'notebook13_cascaded_pipeline.json'
path.write_text(json.dumps(report,indent=2),encoding='utf-8')
print('Saved:',path)
print('Saved:',PROC/'notebook13_test_cascade_top10.csv')

Saved: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\reports\notebook13_cascaded_pipeline.json
Saved: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\data\processed\notebook13_test_cascade_top10.csv


## Interpretation and next steps

- A lower Top-N reduces the **number of DeepFM candidate evaluations in a future conditional implementation**; this notebook does **not** establish measured compute savings.
- Compare `gated_deepfm` vs `ungated_deepfm` to quantify **candidate truncation loss**. Compare `gated_deepfm_mmr` vs `gated_deepfm` for **diversity vs relevance**.
- The current cascade uses the Notebook 09 **15-feature LightGBM**, not the Notebook 11 enhanced 34-feature model. Notebook 11's gain was small and not consistently better across K; a later experiment may swap models with August-only selection.
- Notebook 14: offline replay and error analysis. Notebook 15: actual stage inference timing and p95 latency. Notebook 16: FastAPI serving demo.
- Upload `reports/notebook13_cascaded_pipeline.json` after running all cells.